# Phase 6 — RAG Fraud Analyst Assistant

**Goal:** Build a Retrieval-Augmented Generation (RAG) pipeline that explains *why* a transaction was flagged as fraud in plain English — the way a fraud ops analyst would at Stripe.

### What RAG adds to the pipeline
XGBoost gives you a probability (e.g. 87% fraud). SHAP tells you which features pushed the score up. But neither of these tells an ops agent *what kind of fraud this looks like* in human language. That's what RAG does:

1. **XGBoost** scores the transaction → 87% fraud probability  
2. **SHAP** identifies top risk signals → geo_distance_km, hour_of_day, velocity_24h  
3. **RAG retrieves** the most relevant chunks from your knowledge base → card-testing pattern, CNP fraud signals  
4. **LLM generates** a 2–3 sentence analyst narrative grounded in those retrieved facts  

This mirrors exactly how fraud analyst tools at Stripe, Revolut, and Adyen work.

### Knowledge base
The 4 `.txt` files in `knowledge_base/` are the RAG source — written from your own EDA findings and Stripe domain experience:
- `eda_findings.txt` — fraud patterns found in Phase 1
- `fraud_patterns.txt` — card-testing, CNP, account takeover, geographic anomaly
- `feature_explanations.txt` — what each SHAP feature means in fraud ops terms
- `historical_cases.txt` — example flagged transactions with outcomes

### LLM options
- **Option A (default):** OpenAI GPT-4o-mini — very cheap (~$0.001/call), no extra install needed. Uses `OPENAI_API_KEY` already in `.env`.
- **Option B:** Claude Haiku API — set `ANTHROPIC_API_KEY` in `.env`. Best for Streamlit Cloud deployment.
- **Option C:** Ollama/Mistral — 100% free and local. Requires `ollama serve` + `ollama pull mistral` to be run first.

## Cell 1 — Imports and path setup

Each import plays a specific role in the RAG pipeline:

- **`DirectoryLoader` / `TextLoader`** — LangChain loaders that read `.txt` files into `Document` objects with metadata (source path, content)
- **`RecursiveCharacterTextSplitter`** — splits long documents into overlapping chunks so the LLM's context window is never exceeded
- **`Chroma`** — ChromaDB vector store: stores text as numerical vectors (embeddings) so similarity search is fast
- **`HuggingFaceEmbeddings`** — converts text to vectors using `sentence-transformers/all-MiniLM-L6-v2`, a compact but powerful free model
- **`chromadb`** — the underlying vector database client (ChromaDB persists to disk so you only build it once)

Path constants are set relative to this notebook's location, consistent with how all other notebooks in this project handle paths.

In [1]:
import os
from pathlib import Path

from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter      # langchain 1.x
from langchain_chroma import Chroma                                      # replaces langchain_community.vectorstores.Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate                        # langchain 1.x
from langchain_classic.chains import RetrievalQA                        # langchain 1.x
import chromadb

NOTEBOOK_DIR   = Path('.').resolve()
ROOT           = NOTEBOOK_DIR.parent
KNOWLEDGE_BASE = ROOT / 'knowledge_base'
CHROMA_DB_PATH = ROOT / 'data' / 'chroma_db'

print(f"Knowledge base : {KNOWLEDGE_BASE}")
print(f"ChromaDB path  : {CHROMA_DB_PATH}")
print(f"KB files found : {[f.name for f in KNOWLEDGE_BASE.glob('*.txt')]}")

Knowledge base : C:\Users\34673\OneDrive\IRONHACK BOOTCAMP1\EXERCISES\WEEK8\PROJECT_FRAUD_DETECTION\fraud-detection-project\knowledge_base
ChromaDB path  : C:\Users\34673\OneDrive\IRONHACK BOOTCAMP1\EXERCISES\WEEK8\PROJECT_FRAUD_DETECTION\fraud-detection-project\data\chroma_db
KB files found : ['eda_findings.txt', 'feature_explanations.txt', 'fraud_patterns.txt', 'historical_cases.txt']


## Cell 2 — Load knowledge base documents

**`DirectoryLoader`** scans the `knowledge_base/` folder for `.txt` files and uses `TextLoader` to read each one into a LangChain `Document` object.

A `Document` has two parts:
- `page_content` — the raw text content of the file
- `metadata` — automatically populated with the source file path, so the RAG chain can cite which file a retrieved chunk came from

We load all 4 files as-is here. In the next step we split them into smaller chunks.

In [2]:
loader = DirectoryLoader(
    str(KNOWLEDGE_BASE),
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
documents = loader.load()

print(f"Loaded {len(documents)} documents from knowledge base")
for doc in documents:
    source = Path(doc.metadata['source']).name
    print(f"  - {source:35s}  ({len(doc.page_content):,} chars)")

Loaded 4 documents from knowledge base
  - eda_findings.txt                     (7,424 chars)
  - feature_explanations.txt             (8,343 chars)
  - fraud_patterns.txt                   (7,268 chars)
  - historical_cases.txt                 (8,403 chars)


## Cell 3 — Split documents into chunks

**Why we split:**  
Large language models have a limited context window (the maximum text they can read at once). If we fed the entire knowledge base into every prompt, we'd exceed that limit — and we'd flood the LLM with irrelevant text. Instead, we split documents into small chunks, embed each chunk, and only retrieve the 3 most relevant ones per query.

**`chunk_size=500`** — each chunk is at most 500 characters. This is roughly 100–120 words, a size that fits comfortably in the LLM's context while being specific enough to be useful.

**`chunk_overlap=100`** — adjacent chunks share 100 characters. This prevents a fraud pattern explanation from being cut in half right at the critical line — the overlap ensures context bleeds into the next chunk.

**`separators`** — the splitter tries to break on paragraph boundaries first (`\n\n`), then lines (`\n`), then words (` `), then characters as a last resort. This preserves the structure of our fraud pattern descriptions.

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""],
)
chunks = text_splitter.split_documents(documents)

print(f"Split {len(documents)} documents into {len(chunks)} chunks")
print(f"Average chunk size: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars")
print(f"\nExample chunk (chunk 0):")
print("-" * 60)
print(chunks[0].page_content)
print("-" * 60)
print(f"Source: {Path(chunks[0].metadata['source']).name}")

Split 4 documents into 88 chunks
Average chunk size: 359 chars

Example chunk (chunk 0):
------------------------------------------------------------
FRAUD DETECTION EDA FINDINGS — SPARKOV DATASET
Irish-Lev | Ironhack Data Science & ML Bootcamp | Barcelona 2026
Source: notebooks/01_eda.ipynb

DATASET OVERVIEW
------------------------------------------------------------
Source: eda_findings.txt


## Cell 4 — Create the embedding model

**What embeddings are:**  
An embedding model converts text into a list of numbers (a vector) that captures the *meaning* of the text. Two sentences about card-testing fraud will have vectors that point in a similar direction in vector space — even if they use different words. This enables *semantic search* — finding relevant chunks by meaning, not just exact keyword matches.

**Why `sentence-transformers/all-MiniLM-L6-v2`:**  
- Free and runs locally on CPU
- 384-dimensional output — compact but high quality
- Trained specifically to produce semantically meaningful sentence embeddings
- Already installed in the project venv via `sentence-transformers`

**Keyword search vs semantic search:**  
Keyword: searching "card testing" only finds documents that contain those exact words.  
Semantic: embedding "multiple small transactions" finds documents about card-testing even if they don't use those words — because the meaning is similar.

Note: this is the **same embedding model** used by `app/rag_assistant.py`, so ChromaDB built here works identically in the Streamlit app.

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Smoke test: embed a fraud query and check dimensionality
test_embedding = embeddings.embed_query("card not present fraud at night")
print(f"Embedding model loaded successfully")
print(f"Embedding dimension: {len(test_embedding)}")
print(f"First 5 values: {[round(v, 4) for v in test_embedding[:5]]}")
print(f"\nThis model will embed every chunk in the knowledge base.")
print(f"At query time, the same model embeds the fraud query for similarity search.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully
Embedding dimension: 384
First 5 values: [0.0024, 0.0418, 0.005, 0.0098, 0.0109]

This model will embed every chunk in the knowledge base.
At query time, the same model embeds the fraud query for similarity search.


## Cell 5 — Build ChromaDB vector store

**What ChromaDB is:**  
ChromaDB is a vector database — it stores text chunks alongside their embedding vectors and supports fast nearest-neighbour similarity search. Think of it as a specialised database where the "index" is the embedding space and "WHERE" clauses are replaced with "find me the most similar vectors".

**`Chroma.from_documents()`** — embeds every chunk using our model, stores the (chunk text, embedding, metadata) triples in ChromaDB, and persists the database to disk.

**`persist_directory`** — ChromaDB writes itself to `data/chroma_db/`. After this cell runs once, you never need to rebuild. `app/rag_assistant.py` loads from this path with `Chroma(persist_directory=...)` — no re-embedding, instant startup.

This is why `data/chroma_db/` is in `.gitignore` — it's regenerated locally, not committed to GitHub.

In [5]:
import shutil

rebuilt = False
if CHROMA_DB_PATH.exists():
    try:
        shutil.rmtree(CHROMA_DB_PATH)
        print("Removed existing ChromaDB — rebuilding from scratch")
        rebuilt = True
    except PermissionError:
        print("WARNING: ChromaDB files are locked (Streamlit app is running).")
        print("Loading existing ChromaDB from disk instead — this is fine.")
        print("Stop the Streamlit app and re-run this cell only if you want a full rebuild.")

if rebuilt or not CHROMA_DB_PATH.exists():
    CHROMA_DB_PATH.mkdir(parents=True, exist_ok=True)
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=str(CHROMA_DB_PATH),
    )
    print(f"ChromaDB built with {vectorstore._collection.count()} chunks")
    print(f"Persisted to: {CHROMA_DB_PATH}")
else:
    # Load existing store — same data, no rebuild needed
    vectorstore = Chroma(
        persist_directory=str(CHROMA_DB_PATH),
        embedding_function=embeddings,
    )
    print(f"ChromaDB loaded from disk: {vectorstore._collection.count()} chunks")

size_kb = sum(f.stat().st_size for f in CHROMA_DB_PATH.rglob('*') if f.is_file()) / 1024
print(f"Size on disk: {size_kb:.1f} KB")
print(f"\nvectorstore ready — cells 6 and 7 will work.")

Removed existing ChromaDB — rebuilding from scratch
ChromaDB built with 88 chunks
Persisted to: C:\Users\34673\OneDrive\IRONHACK BOOTCAMP1\EXERCISES\WEEK8\PROJECT_FRAUD_DETECTION\fraud-detection-project\data\chroma_db
Size on disk: 1008.2 KB

vectorstore ready — cells 6 and 7 will work.


## Cell 6 — Test retrieval

Before plugging an LLM in, we test whether the retrieval step actually finds relevant chunks. This is the most important diagnostic — if the wrong chunks are retrieved, no LLM will produce a good explanation.

**`k=3`** — retrieve the top 3 most similar chunks for each query. The RAG chain will inject all 3 into the LLM prompt as context.

**How cosine similarity works (plain English):**  
Both the query and each chunk are converted to vectors. Cosine similarity measures the angle between them — vectors pointing in the same direction (same meaning) score close to 1.0, vectors pointing in opposite directions score close to 0. ChromaDB returns the 3 chunks with the highest similarity scores.

**What to look for:**  
- Query about "2am transaction" → should retrieve hour_of_day explanation + night fraud pattern
- Query about "multiple small transactions" → should retrieve card-testing pattern
- Query about "shopping_net" → should retrieve CNP fraud + category explanation

In [6]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

test_queries = [
    "transaction at 2am far from home large amount",
    "multiple small transactions in 24 hours card testing",
    "shopping_net category online fraud high geo_distance",
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-" * 65)
    docs = retriever.invoke(query)
    for i, doc in enumerate(docs):
        source = Path(doc.metadata['source']).name
        preview = doc.page_content[:160].replace('\n', ' ')
        print(f"  [{i+1}] {source}")
        print(f"       {preview}...")


Query: 'transaction at 2am far from home large amount'
-----------------------------------------------------------------


  [1] eda_findings.txt
       KEY INSIGHT: Geographic distance between the cardholder's registered home address and the merchant location is one of the strongest fraud signals available. A t...
  [2] historical_cases.txt
       EXPLANATION FOR OPS AGENT: Although the geo_distance is high and the amount is elevated, the merchant category is "travel" which is expected to show high distan...
  [3] eda_findings.txt
       Fraud rate spikes sharply between midnight (00:00) and 4am. The high-risk window (0–4am) shows approximately 3-4x the average fraud rate....

Query: 'multiple small transactions in 24 hours card testing'
-----------------------------------------------------------------
  [1] feature_explanations.txt
       WHAT IT IS: The number of transactions on the same card in the 24 hours preceding the current transaction. Engineered in Phase 2 using a rolling time window.  W...
  [2] eda_findings.txt
       CARD-TESTING PATTERN: Fraudsters make many small transactions (under $5-10) 

## Cell 7 — Build the RAG chain

**`RetrievalQA`** — a LangChain chain that wires retrieval and generation together:
1. Takes a query string
2. Calls the retriever → gets top-3 chunks
3. Fills the prompt template with `{context}` (chunks) + `{question}` (query)
4. Sends the filled prompt to the LLM
5. Returns the LLM's response + the source documents

**`chain_type="stuff"`** — "stuff" means all retrieved chunks are stuffed directly into the prompt as one block. This is the simplest approach and works well when `k=3` chunks of 500 chars each (≈1500 chars total) fit comfortably in the LLM context window.

**Prompt design:**  
The prompt explicitly sets the analyst persona ("fraud analyst at Stripe"), specifies the output format ("2–3 sentences, fraud ops terminology"), and separates knowledge base context from the query. This structure prevents the LLM from hallucinating — it's grounded in the retrieved text.

**Three LLM options:**
- **Option A — OpenAI GPT-4o-mini** (default): Cheapest and easiest — your key is already in `.env`. About $0.001 per RAG call.
- **Option B — Claude Haiku API**: Slightly higher quality, same cost order. Set `ANTHROPIC_API_KEY` in `.env`.
- **Option C — Ollama/Mistral**: Fully free and local, but requires installing Ollama (4GB download) and running `ollama serve`.

In [7]:
def _load_env_key(key_name: str) -> str | None:
    """Read an API key from environment or .env file."""
    val = os.getenv(key_name)
    if val:
        return val
    env_path = ROOT / ".env"
    if env_path.exists():
        for line in env_path.read_text(encoding="utf-8").splitlines():
            if line.startswith(f"{key_name}="):
                return line.split("=", 1)[1].strip()
    return None


# Relevance-aware context retrieval 
def get_context(user_question, vectorstore, score_threshold=0.3, k=5):
    """
    Retrieve relevant chunks from ChromaDB using similarity scores.
    Only returns chunks that score ABOVE the threshold — prevents the LLM
    from hallucinating answers from irrelevant chunks.

    If no chunk scores above the threshold, returns a fallback message so
    the LLM knows to answer from its own domain expertise instead.

    Args:
        user_question:   The question to search for
        vectorstore:     ChromaDB Chroma instance
        score_threshold: Minimum relevance score (0-1). Default 0.3.
                         Lower = more permissive. Higher = stricter.
        k:               Max chunks to retrieve. Default 5.

    Returns:
        (docs, formatted_context, source)
        source is "knowledge_base" or "domain_expertise"
    """
    results = vectorstore.similarity_search_with_relevance_scores(
        user_question, k=k
    )

    print("Top relevance scores:")
    for doc, score in results[:5]:
        preview = doc.page_content[:70].replace("\n", " ")
        print(f"  {score:.4f} | {Path(doc.metadata.get('source','?')).name} | {preview}...")

    good_docs = [doc for doc, score in results if score >= score_threshold]

    if good_docs:
        print(f"\nSource: KNOWLEDGE BASE ({len(good_docs)} chunks above threshold {score_threshold})")
        context = "\n".join([
            f"[Source: {Path(d.metadata.get('source','?')).name}]\n{d.page_content}"
            for d in good_docs
        ])
        return good_docs, context, "knowledge_base"

    # No good chunks found — tell the LLM to use its own expertise
    print(f"\nNo chunks above {score_threshold} — LLM will answer from domain expertise")
    fallback_context = (
        "The fraud knowledge base does not contain a direct answer to this question. "
        "Use your broader fraud detection and payment operations expertise to answer."
    )
    return [], fallback_context, "domain_expertise"


# LLM setup 
LLM_OPTION = "openai"   # "openai" | "claude" | "ollama"

if LLM_OPTION == "openai":
    from langchain_openai import ChatOpenAI
    api_key = _load_env_key("OPENAI_API_KEY")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3, openai_api_key=api_key)
    print(f"LLM: OpenAI GPT-4o-mini (key: {'yes' if api_key else 'NO'})")

elif LLM_OPTION == "claude":
    from langchain_anthropic import ChatAnthropic
    api_key = _load_env_key("ANTHROPIC_API_KEY")
    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0.3,
                        anthropic_api_key=api_key)
    print(f"LLM: Claude Haiku (key: {'yes' if api_key else 'NO'})")

else:
    from langchain_community.llms import Ollama
    llm = Ollama(model="mistral", temperature=0.3)
    print("LLM: Ollama/Mistral (local)")


# Prompt 1: Transaction explanation 
EXPLAIN_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""## SYSTEM ROLE
You are a fraud analyst assistant at a payment platform like Stripe or Revolut.
A transaction has been flagged by the ML model and you must explain WHY it was flagged
in plain English that an ops agent can immediately act on.

## TASK
Explain why this specific transaction was flagged as fraud.
Be direct and specific. Use fraud operations terminology.
Keep your response to 2-3 sentences maximum.

## CONTEXT FROM FRAUD KNOWLEDGE BASE
{context}

## FLAGGED TRANSACTION DETAILS
{question}

## ANALYST EXPLANATION (2-3 sentences):""",
)


# Prompt 2: General Q&A
QA_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""## SYSTEM ROLE
You are a senior fraud analyst and ML systems expert with deep knowledge of payment fraud
detection, risk operations, machine learning models, and fraud ops workflows.
You are equivalent to a lead fraud specialist at Stripe, Revolut, or Adyen.

## QUESTION TYPE DETECTION — READ THIS FIRST
Before answering, classify the question:
- TYPE A — CONCEPTUAL: asks how something works, what differentiates X from Y,
  how a model would detect something, what signals indicate something.
  → Answer as an expert explaining a mechanism, methodology, or concept.
  → Use examples. Structure in clear paragraphs. NEVER say "This transaction was flagged".
- TYPE B — TRANSACTION: explicitly describes a specific transaction with a fraud score,
  SHAP values, amount, category etc.
  → Answer as an analyst explaining that specific flagged case.

## GUIDELINES
1. Answer the EXACT question asked — do not reframe it as something else.
2. For TYPE A questions: explain clearly with examples. 2-3 paragraphs.
3. For TYPE B questions: explain the specific transaction in 2-3 sentences.
4. Use the KNOWLEDGE BASE as your primary source.
5. If the knowledge base context says "does not contain a direct answer",
   answer from your broader fraud domain expertise — do not refuse to answer.
6. Never speculate beyond fraud domain knowledge.
7. Format your response in clean, readable prose.

## CONTEXT FROM FRAUD KNOWLEDGE BASE
{context}

## USER QUESTION
{question}

## ANSWER:""",
)


# Build two chains with SEPARATE retrievers 
explain_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
qa_retriever      = vectorstore.as_retriever(search_kwargs={"k": 5})

rag_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=explain_retriever,
    chain_type_kwargs={"prompt": EXPLAIN_PROMPT}, return_source_documents=True,
)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=qa_retriever,
    chain_type_kwargs={"prompt": QA_PROMPT}, return_source_documents=True,
)

print(f"\nExplain chain: {type(llm).__name__} + EXPLAIN_PROMPT (k=3)")
print(f"QA chain:      {type(llm).__name__} + QA_PROMPT (k=5, score-filtered)")
print("get_context() function ready for score-threshold retrieval")


LLM: OpenAI GPT-4o-mini (key: yes)

Explain chain: ChatOpenAI + EXPLAIN_PROMPT (k=3)
QA chain:      ChatOpenAI + QA_PROMPT (k=5, score-filtered)
get_context() function ready for score-threshold retrieval


## Cell 8 — Test the full RAG pipeline end-to-end

This is the moment everything comes together. The query below mimics exactly what `app/rag_assistant.py` sends when a transaction is flagged in the Streamlit app:

1. **Query embedding** — the query string is converted to a vector
2. **Similarity search** — ChromaDB finds the 3 most relevant chunks from the knowledge base
3. **Prompt assembly** — the 3 chunks fill `{context}`, the query fills `{question}`
4. **LLM generation** — the model reads the filled prompt and writes a 2–3 sentence explanation
5. **Return** — `result['result']` is the explanation text; `result['source_documents']` lists which chunks were retrieved

The source documents section tells you *why* the LLM said what it said — full transparency into the RAG chain's reasoning, just like Stripe's internal fraud tooling shows the evidence trail for a flagged transaction.

In [8]:
# Test 1 — Transaction explanation (TYPE B question) 
print("TEST 1 — Transaction explanation")
print("=" * 65)
explain_query = (
    "Transaction flagged at 87.0% fraud probability. "
    "Key SHAP risk signals: geo_distance_km=+2.143, hour_of_day=+1.821, "
    "velocity_24h=+1.504, is_night=+0.932. "
    "Category: shopping_net, Amount: $1240.00, Hour: 02:00, "
    "Distance from home: 840 km. "
    "Why was this transaction flagged as fraud?"
)
result1 = rag_chain.invoke({"query": explain_query})
print(result1["result"])
print("\nSources:", [Path(d.metadata['source']).name for d in result1["source_documents"]])

# Test 2 — Conceptual Q&A using get_context() score filtering 
print("\n\nTEST 2 — Conceptual Q&A (should NOT say \'This transaction was flagged\')")
print("=" * 65)

conceptual_q = (
    "How would the model or app differentiate whether the transaction "
    "is a monthly subscription basis on small amounts or it is from "
    "card testing or a payment bot?"
)

# Use score-filtered retrieval — teacher's approach
docs, context, source = get_context(conceptual_q, vectorstore, score_threshold=0.3, k=5)

# Build the prompt manually for full control
from langchain_openai import ChatOpenAI
import openai

client_openai = openai.OpenAI(api_key=_load_env_key("OPENAI_API_KEY"))

full_prompt = QA_PROMPT.format(context=context, question=conceptual_q)
response = client_openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": full_prompt}],
    temperature=0.3,
    max_tokens=600,
    top_p=0.9,
    frequency_penalty=0.3,
    presence_penalty=0.2,
)
answer = response.choices[0].message.content
print(f"\nSource used: {source.upper()}")
print(f"\n{answer}")

# Test 3 — Another conceptual question 
print("\n\nTEST 3 — Another conceptual question")
print("=" * 65)
conceptual_q2 = "What are the most reliable signals for detecting account takeover fraud?"
docs2, context2, source2 = get_context(conceptual_q2, vectorstore, score_threshold=0.3, k=5)
full_prompt2 = QA_PROMPT.format(context=context2, question=conceptual_q2)
response2 = client_openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": full_prompt2}],
    temperature=0.3, max_tokens=500, top_p=0.9,
    frequency_penalty=0.3, presence_penalty=0.2,
)
print(f"\nSource used: {source2.upper()}")
print(response2.choices[0].message.content)


TEST 1 — Transaction explanation
This transaction was flagged due to a high fraud probability of 87%, primarily because it occurred 840 km away from the cardholder's home, which is significantly outside their normal spending range. Additionally, it was processed at 2 AM, a peak time for automated fraud, and the amount of $1,240 is elevated compared to the cardholder's average spending, indicating potential card-testing behavior.

Sources: ['feature_explanations.txt', 'feature_explanations.txt', 'feature_explanations.txt']


TEST 2 — Conceptual Q&A (should NOT say 'This transaction was flagged')
Top relevance scores:
  0.3756 | eda_findings.txt | CARD-TESTING PATTERN: Fraudsters make many small transactions (under $...
  0.2965 | feature_explanations.txt | WHY IT MATTERS: Transaction amount is a weak standalone predictor (low...
  0.2891 | fraud_patterns.txt | REAL-WORLD CONTEXT (Stripe): Card-testing is extremely common. Fraudst...
  0.2724 | eda_findings.txt | WHY THIS MATTERS OPERATI

## Cell 9 — Integration function for the Streamlit app

This function is the clean interface between the RAG pipeline and the Streamlit app. `app/rag_assistant.py` already implements equivalent functions (`explain_transaction`, `ask_knowledge_base`). This cell defines the same logic inline so you can test it here in the notebook before relying on the module.

**Why wrap it in a function?**  
The Streamlit app calls this once per transaction. The function takes the raw model outputs (fraud score, SHAP dict, transaction dict) and formats them into a natural-language query — the same transformation `explain_transaction()` does in `rag_assistant.py`.

**What each argument means in fraud ops terms:**
- `fraud_score` — XGBoost's predicted probability (e.g. 0.87 = 87% confidence this is fraud)
- `shap_summary` — the top SHAP values: which features are pushing the score up the most (geo_distance is a bigger push than velocity, etc.)
- `transaction_details` — the raw inputs the analyst would see in a case management tool

In [9]:
def get_fraud_explanation(
    fraud_score: float,
    shap_summary: dict,
    transaction_details: dict,
) -> str:
    """
    Build a natural-language fraud explanation from model outputs.
    Called by the Streamlit app for each flagged transaction.

    Args:
        fraud_score:         XGBoost fraud probability, 0.0–1.0
        shap_summary:        {feature_name: shap_value} for top 3–5 features
        transaction_details: {category, amount, hour, geo_distance_km}

    Returns:
        2–3 sentence plain-English analyst narrative
    """
    feat_str = ", ".join(
        f"{k}={v:+.3f}" for k, v in
        sorted(shap_summary.items(), key=lambda x: abs(x[1]), reverse=True)
    )
    query = (
        f"Transaction flagged at {fraud_score:.1%} fraud probability. "
        f"Key SHAP risk signals: {feat_str}. "
        f"Category: {transaction_details.get('category', 'unknown')}, "
        f"Amount: ${transaction_details.get('amount', 0):.2f}, "
        f"Hour: {transaction_details.get('hour', 0):02d}:00, "
        f"Distance from home: {transaction_details.get('geo_distance_km', 0):.0f} km. "
        f"Why was this transaction flagged as fraud?"
    )
    result = rag_chain.invoke({"query": query})
    return result["result"]


# Test with a HIGH-risk scenario (Sparkov card-testing pattern)
explanation = get_fraud_explanation(
    fraud_score=0.9999,
    shap_summary={
        "geo_distance_km": 2.143,
        "hour_of_day": 1.821,
        "velocity_24h": 1.504,
        "is_night": 0.932,
        "log_amt": 0.614,
    },
    transaction_details={
        "category": "shopping_net",
        "amount": 4500.00,
        "hour": 2,
        "geo_distance_km": 840,
    },
)

print("HIGH-RISK EXPLANATION:")
print(explanation)

# Test with a LOW-risk scenario (control case)
low_explanation = get_fraud_explanation(
    fraud_score=0.0033,
    shap_summary={
        "geo_distance_km": -0.412,
        "hour_of_day": -0.208,
        "velocity_24h": -0.156,
    },
    transaction_details={
        "category": "grocery_pos",
        "amount": 52.00,
        "hour": 14,
        "geo_distance_km": 3,
    },
)
print("\nLOW-RISK EXPLANATION:")
print(low_explanation)

HIGH-RISK EXPLANATION:
This transaction was flagged as fraud due to a combination of high-risk factors: it occurred 840 km away from the cardholder's home, which is far outside their normal spending range, and it took place at 2 AM, a peak time for automated fraud. Additionally, the transaction amount of $4,500 is significantly elevated compared to typical spending patterns, especially in the online retail category, which is known for high fraud rates.

LOW-RISK EXPLANATION:
This transaction was flagged due to a combination of factors that raised suspicion despite a low overall fraud probability. The amount of $52.00 is slightly elevated compared to typical grocery spending, and while the transaction occurred at a reasonable hour (2 PM) and the distance from home is only 3 km, the flagged signals indicate potential unusual spending behavior that warrants further review. Recommend investigating the cardholder's recent transaction history for any anomalies.


## Cell 10 — ChromaDB reload function + handoff to Streamlit

**Why this matters:**  
You built ChromaDB once in Cell 5. Every time the Streamlit app starts, it reloads the existing ChromaDB from disk — no re-embedding, no re-building. This is what makes the app fast at startup.

The `load_rag_chain()` function in `app/rag_assistant.py` is identical to what's shown here. The notebook version is for verification — run this cell to confirm the persisted ChromaDB loads cleanly, independently of Cell 5.

**Swapping LLM for Streamlit Cloud:**  
When you deploy to Streamlit Cloud, set `llm_option="claude"` (or keep `"openai"` — both work in the cloud since they're API-based). Add the API key to Streamlit Secrets in the Streamlit Cloud dashboard.

**Data flow summary:**
```
knowledge_base/*.txt  →  [this notebook: Cells 1–5]  →  data/chroma_db/
  →  app/rag_assistant.py  load_rag_chain()
  →  app/app.py Tab 4  (Fraud Analyst Assistant)
```

In [ ]:
def load_rag_chain_from_disk(llm_option: str = "openai"):
    """Load existing ChromaDB from disk and return RAG chains dict."""
    if not CHROMA_DB_PATH.exists():
        print(f"ERROR: ChromaDB not found at {CHROMA_DB_PATH}")
        return None

    emb = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
    vs = Chroma(persist_directory=str(CHROMA_DB_PATH), embedding_function=emb)

    if llm_option == "openai":
        from langchain_openai import ChatOpenAI
        lm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3,
                        openai_api_key=_load_env_key("OPENAI_API_KEY"))
    elif llm_option == "claude":
        from langchain_anthropic import ChatAnthropic
        lm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0.3,
                           anthropic_api_key=_load_env_key("ANTHROPIC_API_KEY"))
    else:
        from langchain_community.llms import Ollama
        lm = Ollama(model="mistral", temperature=0.3)

    explain_chain = RetrievalQA.from_chain_type(
        llm=lm, chain_type="stuff",
        retriever=vs.as_retriever(search_kwargs={"k": 3}),
        chain_type_kwargs={"prompt": EXPLAIN_PROMPT},
        return_source_documents=True,
    )
    qa_chain = RetrievalQA.from_chain_type(
        llm=lm, chain_type="stuff",
        retriever=vs.as_retriever(search_kwargs={"k": 5}),
        chain_type_kwargs={"prompt": QA_PROMPT},
        return_source_documents=True,
    )
    return {"explain": explain_chain, "qa": qa_chain, "vectorstore": vs}


# Verify reload works independently (does NOT re-build ChromaDB)
chains = load_rag_chain_from_disk(llm_option="openai")
if chains:
    count = chains["vectorstore"]._collection.count()
    print(f"ChromaDB loaded from disk: {count} chunks")
    print(f"ChromaDB path : {CHROMA_DB_PATH}")
    print()
    print("Pipeline handoff:")
    print("  knowledge_base/*.txt  →  [this notebook: Cells 1-5]")
    print("  →  data/chroma_db/  (persisted)")
    print("  →  app/rag_assistant.py  load_rag_chain(llm_option='openai')")
    print("  →  app/app.py Tab 4  (Fraud Analyst Assistant)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ChromaDB loaded from disk: 88 chunks
ChromaDB path : C:\Users\34673\OneDrive\IRONHACK BOOTCAMP1\EXERCISES\WEEK8\PROJECT_FRAUD_DETECTION\fraud-detection-project\data\chroma_db

Pipeline handoff:
  knowledge_base/*.txt  →  [this notebook: Cells 1-5]
  →  data/chroma_db/  (persisted)
  →  app/rag_assistant.py  load_rag_chain(llm_option='openai')
  →  app/app.py Tab 4  (Fraud Analyst Assistant)


: 